<h1>LSTM Analysis</h1>

<h2>Description</h2>
<p>Dataset can be downloaded at: </p>
<p>Features</p>

<h2>Environment Set-up</h2>

In [1]:
import os
import sys
# Setting Hadoop home directory for the JVM
os.environ["JAVA_HOME"] = r"C:\Program Files\Java\jdk-21.0.12"
os.environ["SPARK_HOME"] = r"C:\tools\Anaconda3\Lib\site-packages\pyspark"
os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["PATH"] = (
    os.path.join(os.environ["JAVA_HOME"], "bin") + os.pathsep +
    os.path.join(os.environ["HADOOP_HOME"], "bin") + os.pathsep + 
    os.environ["PATH"]
    )

In [ ]:
import regex

#-------------------------------------------------------------------------------
# Data Manipulation Libraries
#-------------------------------------------------------------------------------
#Pandas : Used for data loading, manipulation, cleaning, and tabular data structures 
import pandas as pd
# Regular Expression: Pattern matching and text cleaning
import re


#-------------------------------------------------------------------------------
# Data Visualization Libraries
#-------------------------------------------------------------------------------
import seaborn as sns
import matplotlib.pyplot as plt
from wordcloud import WordCloud
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

#-------------------------------------------------------------------------------
# Mathematical and Statistical Computing Libraries
#-------------------------------------------------------------------------------
import numpy as np
from scipy.stats import pearsonr

#-------------------------------------------------------------------------------
# Natural Language Processing Libraries
#-------------------------------------------------------------------------------
# NLTK: Natural Language Toolkit
import nltk
# English dictionary words and stopwords
from nltk.corpus import words, stopwords
# Tokenize
from nltk.tokenize import word_tokenize
# N-Gram generation
from nltk.util import ngrams
# Language Detection
import fasttext
# Lemmatization
from nltk.stem import WordNetLemmatizer
# VADER sentiment analyzer
from nltk.sentiment import SentimentIntensityAnalyzer
# Delete
from textblob import TextBlob
# VADER negation list
from vaderSentiment.vaderSentiment import NEGATE
#
from gensim.models import Word2Vec

#-------------------------------------------------------------------------------
# Feature Extraction and Teext Vectorization
#-------------------------------------------------------------------------------
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.feature_extraction import text 

#-------------------------------------------------------------------------------
# Topic Modelling
#-------------------------------------------------------------------------------
from sklearn.decomposition import LatentDirichletAllocation

#-------------------------------------------------------------------------------
# Machine Learning Models for Sentiment Classification.
#-------------------------------------------------------------------------------
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Bidirectional, Dropout, Dense
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.backend import K
import optuna

#-------------------------------------------------------------------------------
# Model Evaluation Metrics Libraries
#-------------------------------------------------------------------------------
from sklearn.metrics import classification_report, accuracy_score, ConfusionMatrixDisplay, confusion_matrix, f1_score

#-------------------------------------------------------------------------------
# Model Selection and ValidationLibraries
#-------------------------------------------------------------------------------
# GridSearchCV: Hyperparameter tuning using cross-validation
# StratifiedKFold: Preserves class distribution across folds
from sklearn.model_selection import GridSearchCV, StratifiedKFold
# Traain-Test Split: Splits dataset into training and testing subsets.
from sklearn.model_selection import train_test_split

#-------------------------------------------------------------------------------
# Pipeline Management
#-------------------------------------------------------------------------------
from sklearn.pipeline import Pipeline

#-------------------------------------------------------------------------------
# Notebook Visualization Settings
#-------------------------------------------------------------------------------
# Enables inline plotting in Jupyter notebooks.
%matplotlib inline

# Sets Seaborn dark theme for cleaner plots.
sns.set_style("darkgrid")
# Sets default color palette
sns.set_palette('husl',8)

In [ ]:
import findspark
findspark.init()
import pyspark
from pyspark.ml.feature import Tokenizer, StopWordsRemover, CountVectorizer, IDF, StringIndexer
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.ml.linalg import Vectors, VectorUDT
from pyspark.sql import SparkSession

#jar_path = r"D:\\PostgreSQL\\postgressql\\postgresql-42.7.12.jar"
spark =( SparkSession.builder
        .master("local[*]")
        .config('spark.hadoop.home.dir', r'C:\hadoop')
        .config('spark.jars.packages','org.postgresql:postgresql:42.7.12')
        .appName("LSTM Author Classification")
        .getOrCreate()
)
print('Spark Version:', spark.version)
print('Hadoop Home:', os.environ["HADOOP_HOME"])

Spark Version: 4.2.0
Hadoop Home: C:\hadoop


<h2>Data Extraction and Dataset Creation </h2>

In [3]:
# Set dataset path
# Copy the path from your dataset folder and past here or it wont work
DATASET_PATH = r"C:\Users\lab_services_student\Desktop\PDAN02_POE_PART_01\Authorship_TextAttribution\dataset"
# Read text file
raw_df = (spark.read.format('binaryFile')
          .option('recursiveFileLookup','true')
          .option('pathGlobalFilter','*.txt')
          .load(DATASET_PATH)
          .withColumn("content", F.expr("decode(content, 'UTF-8')"))
          .select("path", "content")
)

In [4]:
# Parsing author and book_id from file path
parsed_df =(
    raw_df
    .withColumn('normal_path',F.regexp_replace('path',r'\\','/'))
    .withColumn('author',F.regexp_extract(F.col("normal_path"), r'/dataset/([^/]+)/', 1))
    .withColumn('file_name',F.regexp_extract(F.col("normal_path"), r'/([^/]+)$', 1))
    .withColumn('book_name',F.regexp_replace(F.col("file_name"), r'\.[^.]+$', ""))
    .filter(F.col('author')!="")
    .filter(F.col("content").isNotNull())
    
)

In [5]:
# Setting up Chunk Size - Earch Row will have 200 words
chunk_size = 200
# Split text into words
words_df = (
    parsed_df
    .withColumn("words",F.split(
        F.trim(F.regexp_replace(F.col("content"), r"\s+"," ")), " ")
    )
    .filter(F.size(F.col("words")) >= chunk_size)
)

In [ ]:
# Create non-overlapping chunks of 200 words
df = (
    words_df
    .withColumn("starts",
            F.sequence(
                F.lit(0),
                F.size(F.col("words")) - chunk_size,
                F.lit(chunk_size)
            )
    )
    .withColumn("chunks",
                F.transform(
                    F.col("starts"),
                    lambda s: F.array_join(F.slice(F.col('words'), s + 1, chunk_size), " ")
                )
    )
    .select("author","book_name", F.explode("chunks").alias("text"))
    .filter(F.length(F.trim(F.col("text"))) > 0)
)

In [ ]:
# Displaying Spark Dataframe
df.printSchema()
df.show(5, truncate=120)

root
 |-- author: string (nullable = true)
 |-- book_name: string (nullable = true)
 |-- text: string (nullable = true)

+-----------+-------------+------------------------------------------------------------------------------------------------------------------------+
|     author|    book_name|                                                                                                                    text|
+-----------+-------------+------------------------------------------------------------------------------------------------------------------------+
|Leo Tolstoy|War and Peace|An Anonymous Volunteer, and David Widger WAR AND PEACE By Leo Tolstoy/Tolstoi CONTENTS BOOK ONE: 1805 CHAPTER I CHAPT...|
|Leo Tolstoy|War and Peace|CHAPTER II CHAPTER III CHAPTER IV CHAPTER V CHAPTER VI CHAPTER VII CHAPTER VIII CHAPTER IX CHAPTER X CHAPTER XI CHAPT...|
|Leo Tolstoy|War and Peace|CHAPTER X CHAPTER XI CHAPTER XII CHAPTER XIII CHAPTER XIV CHAPTER XV CHAPTER XVI CHAPTER XVII CHAPTER XVIII

In [ ]:
# Saving Dataframe as CSV
df.write.mode("overwrite").option('header', True).csv(r"C:\Users\lab_services_student\Desktop\PDAN02_POE_PART_01\Authorship_TextAttribution\dataset\authors.csv")

<h2>Exlporatory Data Analysis</h2>

<h3>Dataset Inspection</h3>

In [ ]:
df.count()

In [ ]:
# Checking for null values
count_null = df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns])
count_null.show()

In [ ]:
# Checking for empty chunks
empty_chunks = df.filter(F.col("text").isNull() | (F.trim(F.col("text")) == ""))
print(f"Empty chunks: {empty_chunks.count()}")

In [ ]:
# Removing a duplicate rows
df = df.dropDuplicates(subset=["text"])
print(f"Number of rows after removing duplicates: {df.count()}")

In [ ]:
# Removing whitespace and special characters
df = df.withColumn("text", F.trim(F.regexp_replace(F.col("text"), r"r\s+", " ")))

In [ ]:
# Remving any redisual non-printable characters
df = df.withColumn("text", F.regexp_replace(F.col("text"), r"[^\x20-\x7E]", ""))

<h3>Class Balance</h3>

In [ ]:
# Count the number of chunks per author
df.groupBy('author').count().orderBy(F.desc('count')).show(truncate=False)

In [ ]:
# Count Chunks per Author
author_counts =(
    df.groupBy("author").agg(
    F.count("*").alias("num_chunks"),
    F.countDistinct("book_name").alias("num_books")
    )
    .orderBy(F.desc("num_chunks"))
)
author_counts.show(truncate=False)

#Compute imbalance ratio
counts_pd = author_counts.toPandas()
imbalance_ratio = counts_pd["num_chunks"].max() / counts_pd["num_chunks"].min()
print(f"Imbalance Ratio: {imbalance_ratio:.2f}")

+-------------------+-----+
|author             |count|
+-------------------+-----+
|Leo Tolstoy        |8526 |
|Fyodor Dostoyevsky |7530 |
|Jonathan Swift     |6209 |
|Herman Melville    |6038 |
|Nathaniel Hawthorne|5358 |
|Jack London        |5329 |
|Arthur Conan Doyle |3496 |
|Bernard Shaw       |2463 |
+-------------------+-----+



<h3>Text Length and Vocabilary Statistics</h3>

In [ ]:
# Calculate Text Length Statistics
length_stats = df.select(
    F.length("text").alias("char_count"),
    F.size(F.split("text", " ")).alias("word_count")
)

length_stats.select(
    F.mean("char_count").alias("main_chars"),
    F.stddev("char_count").alias("std_chars"),
    F.min("char_count").alias("min_chars"),
    F.max("char_count").alias("max_chars"),
    F.mean("word_count").alias("mean_words"),
    F.stddev("word_count").alias("std_words")
    
).show()

Number of rows


44949

In [ ]:
# Vocabulary size per author
tokenizer = RegexTokenizer(inputCol="text", outputCol="tokens", pattern="\\w+")
tokenized_df = tokenizer.transform(df)

voc_per_author = (tokenized_df
                  .select("author",F.explode("tokens").alias("word"))
                  .groupBy("author")
                  .agg(F.countDistinct("word").alias("vocab_size"))
                  .orderBy(F.desc("vocab_size"))
)
voc_per_author.show(truncate=False)

In [ ]:
global_vocab = (tokenized_df.selct(F.explode("tokens").alias("word"))
               .distinct()
               .count()
)
print(f"Global vocabulary size = {global_vocab}")

<h3>Data Visualisation<h3>

<h4>Word Cloud</h4>

In [ ]:
authors = [row["author"] for row in df.select("author").distinct().collect()]
for author in authors:
    # Aggregate word frequencies for top 200 words
    freq_df = (
        tokenized_df
        .filter(F.col("author") == author)
        .select(F.explode("tokens").alias("word"))
        .groupBy("word").count()
        .orderBy(F.desc("count"))
        .limit(200)
        .toPandas
    )
    freq_dict = dict(zip(freq_df["word", freq_df["count"]]))
    wcloud = WordCloud(width=800,height=400,background_color="white").generate_from_frequencies(freq_dict)

    plt.figure(figsize=(12,6))
    plt.imshow(wcloud, interpolation='bilinear')
    plt.axis("off")
    plt.show()

<h3>Box Plots and Histograms</h3>

In [ ]:
length_pd = (df
             .withColumn("char_count", F.length("text"))
             .withColumn("word_count", F.size(F.split("text", " ")))
             .select("author", "char_count", "word_count")
             .toPandas()
)

fig, axes =plt.subplots(1, 2, figsize=(14, 5))
sns.boxplot(data=length_pd, x="author", y="word_count", ax=axes[0])
axes[0].set_title("Word Count Distribution per Author")
axes[0].tick_params(axis="x", rotation=45)

sns.histplot(data=length_pd, x="char_count",hue="author", bins=30, ax=axes[1])
axes[1].set_title("Character Count Distribution")
plt.tight_layout()
plt.show()

<h3>PCA and t-SNE</h3>

In [ ]:
#Training a Word2Vec
word2vec = Word2Vec(vectorSize=100,minCount=5,inputCol="tokens",seed=42)
w2v_model = word2vec.fit(tokenized_df)
tokenized_df = word2vec.transform(tokenized_df)

In [ ]:
#Calculating word vectors average
def avg_vectors(vectors):
    if not vectors:
        return Vectors.dense(([0.0]*100))
    arr = np.array([v.toArray() for v in vectors])

avg_udf = F.udf(avg_vectors, VectorUDT())
#Getting Document Embeddings
doc_emb_df = tokenized_df.withColumn("doc_vector", avg_udf(F.col("word_vectores")))

: 

In [ ]:
# Converting to Pandas for data visualization
emb_pd = doc_emb_df.select("author","doc_vector").toPandas()
X = np.vstack(emb_pd["doc_vecter"].apply(lambda v: v.toArray()).values)
Y = emb_pd["author"].values

In [ ]:
# Computing PCA
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X)

plt.figure(figsize=(10, 7))
for author in np.unique(Y):
    mask = Y == author
    plt.scatter(X_pca[mask,0], X_pca[mask, 1], label=author, alpha=0.5, s=1.0)

plt.legend()
plt.title("PCA of Document Embeddings")
plt.show()

In [ ]:
# Computing TSNE
X_tsne = TSNE(n_components=2, perplexity=30, random_state=42).fit_transform(X)
plt.figure(figsize=(10,7))
for author in np.unique(y):
    mask = y == author
    plt.scatter(X_tsne[mask,0], X_tsne[mask, 1], label=author, alpha=0.5, s=1.0)
    
plt.legend()
plt.title("t-SNE of Document Embeddings")
plt.show()

<h2>Feature Selectiona and Engineering</h2>

In [ ]:
#RegexTokenizer with lowecasing
tokenizer = RegexTokenizer(inputCol="text", outputCol="tokens",pattern="\\w+", toLowercase=True)
df = tokenizer.transform(df)
df.select("author","book_name","tokens").show(5, truncate=80)

In [ ]:
#Building a vocabulary with Index Mapping
vocab_df = (
    df
    .select(F.explode("tokens").alias("word"))
    .groupBy("word").count()
    .filter(F.col("count") >= 5)
)
# Assigning integer index to each word
vocab_df = vocab_df.withColumn("word_id", F.row_number().over(Window.orderBy(F.desc("count"))-1))
vocab_size = vocab_df.count
print(f"Vocabulary size : {vocab_size}")

<h3>GloVe Embedding Matrix</h3>

In [ ]:
# Loading GolVe vectors
glove_schema = "word string, " + ", ".join([f"v{i} float" for i in range(300)])
# Do not forget to puth model path here
glove_df = spark.read.csv("", sep=" ", schema=glove_schema)

In [ ]:
# Joining vocabularu with GloVe
vocab_glove = vocab_df.join(glove_df, on="word", how="left")
# Count coverage
coverage = vocab_glove.filter(F.col("v0").isNotNull()).count()
g_avg = coverage/vocab_size
g_avg_pct = 100*coverage/vocab_size
print(f"Glove Coverage: {g_avg} ({g_avg_pct:.1f}%) ")

In [ ]:
# Collecting embedding matrix to driver
#Sorting by word_id to get correct index order
vocab_glv_sorted = vocab_glove.orderBy("word_id").toPandas()
np.random.seed(42)
embedding_matrix = np.zeros((vocab_size, 300), dtype=np.float32)

for i, row, in vocab_glv_sorted.iterrows():
    wid = int(row["word_id"])
    if pd.notna(row["v0"]):
        embedding_matrix[wid] = row[[f"v {i}" for i in range(300)]].values.astype(np.float32)
    else:
        # OOV: random normal initialisation
        embedding_matrix[wid] = np.random.normal(0, 0.05, 300).astype(np.float32)

print(f"Embedding matrix shape: {embedding_matrix.shape}")

<h3>Converting Tokens to Index Sequences</h3>

In [ ]:
word_to_id = {row["word"]: row["word_id"] for row in vocab_df.collect()}

#UDF to map tokens to indeces
def tokens_to_ids(tokens):
    return [word_to_id.get(t, 0) for t in tokens] # 0 = <UNK> or <PAD>

tokens_to_ids_udf = F.udf(tokens_to_ids, ArrayType(IntegerType()))
# Functon to Pad or Truncate SEQ_LEN = 200

SEQ_LEN=200
def pad_or_truncate(ids, seq_len=SEQ_LEN):
    if len(ids) >= seq_len:
        return ids[:seq_len]
    return ids + [0] * (seq_len - len(ids))

pad_udf = F.udf(pad_or_truncate, ArrayType(IntegerType()))

df = (
    df
    .withColumn("tokens_ids", tokens_to_ids_udf(F.col("tokens")))
    .withCOlumn("sequence", pad_udf(F.col("tokens")))
)
df.select("author","book_name","sequence").show(5, truncate=80)

<h3>Author Label Encoding</h3>

In [ ]:
label_indexer = StringIndexer(inputCol="author", outputCol="author_index", handleInvalid="skip")
df = label_indexer.fit(df).fit_transform(df)
#Saving label mapping
label_map = df.select("author","author_index").distinct().orderBy("author_index").toPandas()
print(label_map)
NUM_AUTHORS = label_map.shape[0]

<h2>Model Trainng</h2>

<h3>Data Splitting</h3>

In [ ]:
# Gettin unique values
books_df = df.select("author","book_name").distinct()
# Splitting data for each author
# Splitting books in 70/10/20 -> training/validation/test
train_books, val_books, test_books = [], [], []
for row in books_df.groupBy("author").agg(F.collect_list("book_name").alias("books")).collect():
    author = row["author"]
    books = sorted(row["books"])
    
    n = len(books)
    n_train = max(1, int((0.7*n)))
    n_val = max(1,int(0.1*n)) if n >= 3 else 0
    n_test = n-n_train-n_train
    
    if n == 1:
        train_books.append((author, books[0]))
        
    elif n == 2:
        train_books.append((author, books[0]))
        test_books.append((author, books[1]))
        
    else:
        for b in books[:n_train]:
            train_books.append((author, b))
        
        for b in books[n_train:n_train + n_val]:
            val_books.append((author, b))
        
        for b in books[n_train + n_val]:
            test_books.append((author, b))

In [ ]:
# Creating DataFrames
train_books_df = spark.createDataFrame(train_books, ["author", "book_name"])
val_books_df = spark.createDataFrame(val_books, ["author", "book_name"])
test_books_df = spark.createDataFrame(test_books, ["author", "book_name"])

train_df = df.join(train_books_df, on=["author","book_name"], how="inner")
val_df = df.join(val_books_df, on=["author","book_name"], how="inner")
test_df = df.join(test_books_df, on=["author","book_name"], how="inner")

print(f"Train: {train_df.count()} chunks")
print(f"Validation: {val_df.count()} chunks")
print(f"Test: {test_df.count()} chunks")

In [ ]:
# Checking author balance
for name, df in [("Train",train_df), ("Val", val_df), ("Test", test_df)]:
    print(f"\n--- {name} ---")
    df.groupBy("author").count().orderBy("author").show(truncate=False)

In [ ]:
def macro_f1(y_true, y_pred):
    #y_true: (batch,) integer labels
    #y_pred: (batch, num_classes) softmax probabilities
    y_pred = K.cas(K.armgax(y_pred, axis=1), "int32")
    y_true = K.cast(K.reshape(y_true, (-1,)), "int32")
    
    f1s = []
    
    for c in range(NUM_AUTHORS):
        tp = K.sum(K.cast((y_true == c) & (y_pred == c), "float32"))
        fp = K.sum(K.cast((y_true != c) & (y_pred == c), "float32"))
        fn = K.sum(K.cast((y_true == c) & (y_pred != c), "float32"))
        precision = tp/(tp + fp + K.epsilon())
        recall = tp/(tp + fn + K.epsilon())
        f1 = 2*precision*recall/(precision+recall+K.epsilon())
        f1s.append(f1)
    return K.mean(K.stack(f1s))

In [ ]:
# Callback that calcuates sklearn macro-F1 on val set
class MacroF1Callback(Callback):
    def __inint__(self, x_val, y_val, name="val_macro_f1"):
        self.x_val = x_val
        self.y_val = y_val
        self.name = name
    
    def on_epoch_end(self, epoch, logs=None):
        logs =logs if logs is not None else {}
        y_pred = self.model.predictive(self.X_val, verbose=0).argmax(axis=1)
        logs[self.name] = f1_score(self.y_val, y_pred, average="macro")

<h3>Training: Simple Model</h3>

In [ ]:
#Converting Dataframes to Pandas Dataframes
train_pd = train_df.select("Sequence", "author_index").toPandas()
val_pd = val_df.select("Sequence", "author_index").toPandas()
test_pd = test_df.select("Sequence", "author_index").toPandas()
# Creating X and Y variables
X_train = np.vstack(train_pd["sequence"].apply(lambda x: np.array(x)).values())
Y_train = train_pd["author_index"].values()

X_val= np.vstack(train_pd["sequence"].apply(lambda x: np.array(x)).values())
Y_val = train_pd["author_index"].values()


X_test = np.vstack(train_pd["sequence"].apply(lambda x: np.array(x)).values())
Y_test = train_pd["author_index"].values()

In [ ]:
model = Sequential([
    Embedding(
        input_dim=vocab_size,
        output_dim=300,
        weights=[embedding_matrix],
        input_length=SEQ_LEN,
        trainable=True
    ),
    Bidirectional([LSTM(128, return_sequences=False)]),
    Dropout(0.3),
    Dense(NUM_AUTHORS, activation="softmax")
])

model.compile(loss="sparse_categorical_crossentropy",optimizer="adam",metrics=["accuracy", macro_f1])
model.summary()

In [ ]:
callbacks = [
    MacroF1Callback(X_val, Y_val),
    EarlyStopping(monitor="val_macro_f1", mode="max", patience=3,restore_best_weights=True),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6)

]
history = model.fit(X_train, Y_train, 
                    validation_data=(X_val, Y_val),
                    epochs=20, batch_size=64,
                    callbacks=callbacks)

In [ ]:
print("History Keys: ", list(history.history.keys()))

In [ ]:
fig, axes = plt.subplot(1, 3, figsize=(18,6))

axes[0].plot(history.history["loss"], label="Train")
axes[0].plot(history.history["val_loss"], label="Val")
axes[0].set_title("Loss")
axes[0].legend()

axes[1].plot(history.history["accuracy"], label="Train")
axes[1].plot(history.history["val_accuracy"], label="Val")
axes[1].set_title("Accuracy")
axes[1].legend()

axes[2].plot(history.history["macro_f1"], label="Train (batch)")
axes[2].plot(history.history["val_macro_f1"], label="Val (sklearn)")
axes[2].set_title("Macro-F1")
axes[2].legend()

plt.tight_layout()
plt.show()

<h3>Model Optimisation</h3>

In [ ]:
def build_model(trial):
    emb_dim = trial.suggest_categorical("emb_dim", [100, 200, 300])
    hidden = trial.suggest_categorical("hidden_units", [120,256,300, 512])
    n_layers = trial.suggest_int("n_lstm_layers", 1, 2)
    emb_drop = trial.suggest_float("emb_droput", 0.2, 0.5, step=0.1)
    lstm_drop = trial.suggest_float("lstm_droput", 0.2, 0.4, step=0.1)
    lr = trial.suggest_categorical("learning_rate", [0.001, 0.01, 0.1])
    batch_size = trial.suggest_categorical("batch_size", [16, 32, 64, 128])
    
    model = Sequential()
    # Using pre-trined GloVe only if emb_dim==300
    if emb_dim == 300:
        model.add(Embedding(vocab_size, emb_dim,weights=[embedding_matrix], input_length=SEQ_LEN, trainable=True))
    else:
        model.add(Embedding(vocab_size, emb_dim, input_length=SEQ_LEN, trainable=True))
    
    model.add(Dropout(emb_drop))

    for i in range(n_layers):
        return_seq = (i < n_layers-1)
        model.add(Bidirectional(LSTM(hidden, retun_sequences=return_seq)))
        model.add(Dropout(lstm_drop))
    
    model.add(Dense(NUM_AUTHORS, activation="softmax"))
    
    model.compile(
        loss="sparse_categorical_crossentropy",
        optimizer=Adam(learning_rate=lr, clipnorm=1.0),
        metrics=["accuracy",macro_f1]
    )
    
    return model, batch_size

In [ ]:
def objective(trial):
    model, batch_size = build_model(trial)
    
    callbacks = [
        EarlyStopping(monitor="val_macro_f1", mode="max", patience=3, restore_best_weights=True),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6),
        MacroF1Callback(X_val, Y_val)
    ]
    
    history = model.fit(
        X_train, Y_train,
        validation_data=(X_val, Y_val),
        epochs=15,
        batch_size=batch_size,
        callbacks=callbacks,
        verbose=0
    )
    #Return best validaion loss
    return max(history.history["val_macro_f1"])

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=30)

print("Best Trial: ")
print(study.best_trial.params)
print(f"Best val loss: {study.best_trial.value}")

In [ ]:
# Getting best parameters
best_params = study.best_trial.params
# Getting best epochs
best_epochs = [
    t.user_attrs.get("best_epoch", 10)
    for t in study.trials if t.value is not None
]
n_epochs= int(np.median(best_epochs)) if best_epochs else 10
# Rebuilding model with best parameters
final_model, best_batch = build_model(optuna.trial.FixedTrial(best_params))

# Combining train and validation for the final training
X = np.vstack([X_train, X_val])
Y = np.concatenate([Y_train, Y_val])
# Splitting the dat
X_train, X_val, Y_train, Y_val = train_test_split(X,Y, test_size=0.1, stratify=Y, random_state=42)

callbacks =[
    MacroF1Callback(X,Y),
    EarlyStopping(monitor="val_macro_f1", mode="max", patience=3, restore_best_weights=True)
]



final_model.fit(X_train, Y_train,validation_data=(X_val, Y_val), epochs=n_epochs,  batch_size=best_batch,callbacks=callbacks, verbose=1)

<h3>Model Evaluation</h3>

In [ ]:
y_pred_proba = final_model.predict(X_test)
y_pred = y_pred_proba.argmax(axis=1)
author_names= label_map.sort_values("author_index")["author"].tolist()

print("="*60)
print("Classificatio Report")
print("="*60)
classification_report(Y_test, y_pred, target_names=author_names, digits=4)